# Trump Cleaned and saved

In [130]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from wordcloud import wordcloud
from textblob import TextBlob
# from tensorflow.keras.preprocessing.text import Tokenizer
from nltk.sentiment.util import *

nltk.download('stopwords')
nltk.download('wordnet')
lm = WordNetLemmatizer()

    

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ghazalehzehtab/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ghazalehzehtab/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Start new Section with cleaned data

In [131]:
biden_tweets = pd.read_csv("biden_US_cleaned.csv")
trump_tweets = pd.read_csv("trump_US_cleaned.csv")

print(biden_tweets.shape)
biden_tweets.head()


(179575, 4)


,user_id,likes,retweet_count,cleantext
0,19329393.0,73.0,4.0,joebiden enlisting list star power help close ...
1,19329393.0,226.0,14.0,celebs excited america voted blue congrats pre...
2,17169320.0,717.0,55.0,campaigning across battleground state ran grea...
3,21586418.0,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...
4,11912362.0,6748.0,919.0,news outlet projected joebiden secured elector...


In [132]:
trump_tweets.head()

,user_id,likes,retweet_count,cleantext
0,19329393.0,206.0,12.0,without doubt fourseasons debacle best part do...
1,15754281.0,56.0,16.0,president donald trump youngest daughter tiffa...
2,26642006.0,695.0,291.0,woah read article president debt donald trump ...
3,17446621.0,149.0,15.0,latest episode snl tackled donaldtrump loss co...
4,17446621.0,83.0,10.0,nataliemorales explains latinx voter supported...


In [133]:
print(trump_tweets.shape)

(209928, 4)


In [134]:

def user_activity(dataframe):
    tweet_counts = dataframe.groupby('user_id').size()
    labels = tweet_counts.apply(lambda x: 'active' if x > 5 else 'less active')
    dataframe['user_activity'] = dataframe['user_id'].map(labels)
    return dataframe

biden_tweets_activity = user_activity(biden_tweets)
trump_tweets_activity = user_activity(trump_tweets)
biden_tweets_activity.shape

(179575, 5)

In [135]:
biden_tweets_activity[biden_tweets_activity['user_activity'] == 'active'].head()

,user_id,likes,retweet_count,cleantext,user_activity
7,17446621.0,406.0,40.0,victory speech joebiden said understood trump ...,active
8,17446621.0,272.0,18.0,gabrielleunion reacted news joebiden taken lea...,active
9,17446621.0,515.0,38.0,hollywood star cheer joebiden take lead pennsy...,active
10,17446621.0,186.0,15.0,gabrielle union kerry washington eva longoria ...,active
11,17446621.0,73.0,1.0,jimmykimmel offered view last day gone close r...,active


In [136]:
print("Biden active user",biden_tweets_activity[biden_tweets_activity['user_activity'] == 'active'].shape)
print("Biden less active user",biden_tweets_activity[biden_tweets_activity['user_activity'] == 'less active'].shape)

Biden active user (96705, 5)
Biden less active user (82870, 5)


In [137]:
print("Trump active user",trump_tweets_activity[trump_tweets_activity['user_activity'] == 'active'].shape)
print("Trump less active user",trump_tweets_activity[trump_tweets_activity['user_activity'] == 'less active'].shape)

Trump active user (144606, 5)
Trump less active user (65322, 5)


In [138]:
# import matplotlib.pyplot as plt
# 
# # Assuming your DataFrame is named biden_tweets_activity
# # Assuming 'user_activity' column contains activity levels
# 
# # Filter DataFrame for active and less active users
# active_users = biden_tweets_activity[biden_tweets_activity['user_activity'] == 'active']
# less_active_users = biden_tweets_activity[biden_tweets_activity['user_activity'] == 'less active']
# 
# # Plotting histogram
# plt.hist([active_users['user_activity'], less_active_users['user_activity']], 
#          color=['blue', 'red'], 
#          label=['Active Users', 'Less Active Users'])
# 
# # Customize labels and title
# plt.xlabel('Some Column')
# plt.ylabel('Frequency')
# plt.title('Comparison of Active and Less Active Users')
# plt.legend()
# 
# # Show plot
# plt.show()


In [139]:
def getsubjectivity(text):
    return TextBlob(text).sentiment.subjectivity
def getpolarity(text):
    return TextBlob(text).sentiment.polarity
def getAnalysis(score):
    if score <0:
        return 'negative'
    elif score == 0:
        return 'neutral'
    else:
        return 'positive'

In [140]:
def sentiment_analysis(tweets):
    tweets['subjectivity'] = tweets['cleantext'].apply(getsubjectivity)
    
    tweets['polarity'] = tweets['cleantext'].apply(getpolarity)
    
    tweets['analysis'] = tweets['polarity'].apply(getAnalysis)
    return tweets
#print("This is trump tweets analysis", trump_tweets.analysis.value_counts(normalize=True) * 100)


In [141]:
def check_and_convert(value):
    if isinstance(value, float):  # Check if the value is a float
        return str(value)         # Convert float to string
    else:
        return value              # Return value unchanged if not a float

# Apply the function to the DataFrame column
trump_tweets_activity['cleantext'] = trump_tweets_activity['cleantext'].apply(lambda x: check_and_convert(x))


In [142]:
trump_df = sentiment_analysis(trump_tweets_activity)
trump_df.head()

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
0,19329393.0,206.0,12.0,without doubt fourseasons debacle best part do...,less active,0.483333,0.333333,positive
1,15754281.0,56.0,16.0,president donald trump youngest daughter tiffa...,less active,0.000000,0.000000,neutral
2,26642006.0,695.0,291.0,woah read article president debt donald trump ...,less active,0.000000,0.000000,neutral
3,17446621.0,149.0,15.0,latest episode snl tackled donaldtrump loss co...,active,0.800000,-0.033333,negative
4,17446621.0,83.0,10.0,nataliemorales explains latinx voter supported...,active,0.627778,-0.050000,negative


In [143]:
biden_df = sentiment_analysis(biden_tweets_activity)
biden_df.head()

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
0,19329393.0,73.0,4.0,joebiden enlisting list star power help close ...,less active,0.0000,0.00000,neutral
1,19329393.0,226.0,14.0,celebs excited america voted blue congrats pre...,less active,0.6625,0.49375,positive
2,17169320.0,717.0,55.0,campaigning across battleground state ran grea...,less active,0.7500,0.80000,positive
3,21586418.0,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...,less active,0.1000,0.00000,neutral
4,11912362.0,6748.0,919.0,news outlet projected joebiden secured elector...,less active,0.0000,0.00000,neutral


In [144]:
biden_df.head(10)

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
0,19329393.0,73.0,4.0,joebiden enlisting list star power help close ...,less active,0.000000,0.000000,neutral
1,19329393.0,226.0,14.0,celebs excited america voted blue congrats pre...,less active,0.662500,0.493750,positive
2,17169320.0,717.0,55.0,campaigning across battleground state ran grea...,less active,0.750000,0.800000,positive
3,21586418.0,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...,less active,0.100000,0.000000,neutral
4,11912362.0,6748.0,919.0,news outlet projected joebiden secured elector...,less active,0.000000,0.000000,neutral
5,11912362.0,8855.0,784.0,president elect joebiden message http co mvsdl...,less active,0.900000,0.800000,positive
6,11912362.0,5909.0,806.0,presidenttrump release statement following pre...,less active,0.466667,0.533333,positive
7,17446621.0,406.0,40.0,victory speech joebiden said understood trump ...,active,0.750000,-0.750000,negative
8,17446621.0,272.0,18.0,gabrielleunion reacted news joebiden taken lea...,active,0.607143,0.417143,positive
9,17446621.0,515.0,38.0,hollywood star cheer joebiden take lead pennsy...,active,0.000000,0.000000,neutral


In [145]:
sorted_biden = biden_df.sort_values(by=['likes', 'retweet_count'], ascending=False)
sorted_biden.head()

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
102,139823781.0,165702.0,17652.0,tonight woman audience trump town hall began q...,active,0.5125,0.483333,positive
19865,38362690.0,74528.0,20615.0,america assembled joebiden kamalaharris barack...,less active,0.0000,0.000000,neutral
94,139823781.0,59403.0,7489.0,god bette scared god fascist whitehouse racist...,active,0.4500,0.080000,positive
90,139823781.0,59345.0,2838.0,wonderful speech tonight president vice presid...,active,0.6800,0.522500,positive
92,139823781.0,45540.0,6133.0,god bette scared fascist whitehouse racist cri...,active,0.5625,0.100000,positive


In [146]:
biden_df.groupby('user_id')['polarity'].mean()

user_id
1.081000e+03    0.000000
2.654000e+03    0.083333
3.652000e+03    0.000000
4.358000e+03    0.000000
5.608000e+03    0.000000
                  ...   
1.325492e+18    0.000000
1.325502e+18    0.500000
1.325533e+18    0.250000
1.325548e+18   -0.255556
1.325550e+18    0.566667
Name: polarity, Length: 61201, dtype: float64

In [147]:
biden_df.analysis.value_counts()

analysis
neutral     83859
positive    65654
negative    30062
Name: count, dtype: int64

In [148]:
trump_df.analysis.value_counts()

analysis
neutral     90575
positive    71600
negative    47753
Name: count, dtype: int64

In [149]:
sorted_trump = trump_df.sort_values(by=['likes', 'retweet_count'], ascending=False)
sorted_trump.head(1000)

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis
22756,38362690.0,74084.0,20491.0,america assembled joebiden kamalaharris barack...,less active,0.000000,0.000000,neutral
216,47293791.0,35449.0,6629.0,call california joe biden minute poll closed e...,active,0.250000,-0.183333,negative
1179,212674482.0,25987.0,5986.0,white house still released health care plan mi...,less active,0.000000,0.000000,neutral
133,139823781.0,23225.0,1851.0,trump want throw military ballot bone spur,active,0.100000,-0.100000,negative
215,47293791.0,23037.0,2652.0,black trump showed election electionday electi...,active,0.433333,-0.166667,negative
...,...,...,...,...,...,...,...,...
1649,299273962.0,244.0,98.0,life home line make snide comment raking fores...,active,0.000000,0.000000,neutral
864,14179819.0,244.0,31.0,supporter rallied president donald trump n j t...,less active,0.000000,0.000000,neutral
1722,33529092.0,243.0,201.0,privilege living nation men made huge sacrific...,active,0.850000,0.400000,positive
695,499073990.0,243.0,79.0,trump worst job president u history trumpfaile...,active,1.000000,-1.000000,negative


In [150]:
sorted_biden['interest'] = (0.4 * sorted_biden['likes']) +  (0.6 * sorted_biden['retweet_count'])
sorted_biden.head()

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis,interest
102,139823781.0,165702.0,17652.0,tonight woman audience trump town hall began q...,active,0.5125,0.483333,positive,76872.0
19865,38362690.0,74528.0,20615.0,america assembled joebiden kamalaharris barack...,less active,0.0000,0.000000,neutral,42180.2
94,139823781.0,59403.0,7489.0,god bette scared god fascist whitehouse racist...,active,0.4500,0.080000,positive,28254.6
90,139823781.0,59345.0,2838.0,wonderful speech tonight president vice presid...,active,0.6800,0.522500,positive,25440.8
92,139823781.0,45540.0,6133.0,god bette scared fascist whitehouse racist cri...,active,0.5625,0.100000,positive,21895.8


In [152]:
import jenkspy

num_bins = 5

breaks = jenkspy.jenks_breaks(sorted_biden['interest'], n_classes= num_bins)
print(breaks)
# Bin the 'interest' column based on the breaks


[0.0, 686.6, 4252.8, 12832.600000000002, 42180.2, 76872.0]


In [ ]:
# import dask.dataframe as dd
# import dask.array as da
# import dask.distributed
# import jenkspy
# 
# # Assuming 'sorted_biden' is a Dask DataFrame
# # If not, you can convert your data to a Dask DataFrame
# # by using dd.from_pandas() or dd.read_csv() depending on your data source.
# 
# # Define the number of partitions based on your dataset size and available resources
# num_partitions = 10
# num_bins = 2
# # Partition the DataFrame
# sorted_biden_partitioned = sorted_biden.interest(npartitions=num_partitions)
# 
# # Function to calculate natural breaks for each partition
# def calculate_breaks(partition):
#     return jenkspy.jenks_breaks(partition['interest'], n_classes= num_bins)
# 
# # Apply the function to each partition
# partition_breaks = sorted_biden_partitioned.map_partitions(calculate_breaks)
# 
# # Aggregate the results
# final_breaks = partition_breaks.compute()
# 
# print(final_breaks)


In [153]:
sorted_biden['interest_category'] = pd.cut(sorted_biden['interest'], bins=breaks, labels=range(1, num_bins+1), include_lowest=True)
sorted_biden.head(1000)

,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis,interest,interest_category
102,1.398238e+08,165702.0,17652.0,tonight woman audience trump town hall began q...,active,0.512500,0.483333,positive,76872.0,5
19865,3.836269e+07,74528.0,20615.0,america assembled joebiden kamalaharris barack...,less active,0.000000,0.000000,neutral,42180.2,4
94,1.398238e+08,59403.0,7489.0,god bette scared god fascist whitehouse racist...,active,0.450000,0.080000,positive,28254.6,4
90,1.398238e+08,59345.0,2838.0,wonderful speech tonight president vice presid...,active,0.680000,0.522500,positive,25440.8,4
92,1.398238e+08,45540.0,6133.0,god bette scared fascist whitehouse racist cri...,active,0.562500,0.100000,positive,21895.8,4
...,...,...,...,...,...,...,...,...,...,...
756,2.858792e+07,278.0,81.0,people think voting guy vote decency election ...,active,0.000000,0.000000,neutral,159.8,1
2640,4.439796e+07,278.0,64.0,joebiden celebrated abc right donaldtrump inte...,active,0.642857,0.317857,positive,149.6,1
12291,2.095488e+07,278.0,54.0,breaking joebiden take lead georgia vote trump...,active,0.650000,0.400000,positive,143.6,1
676,1.899340e+07,278.0,35.0,breaking abc news characterize joe biden appar...,active,0.412500,0.412500,positive,132.2,1


In [154]:
num_bins = 5

breaks = jenkspy.jenks_breaks(sorted_biden['likes'], n_classes= num_bins)
print(breaks)
sorted_biden['likes_category'] = pd.cut(sorted_biden['likes'], bins=breaks, labels=range(1, num_bins+1), include_lowest=True)
sorted_biden.head(1000)

[0.0, 1498.0, 8855.0, 28696.0, 74528.0, 165702.0]


,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis,interest,interest_category,likes_category
102,1.398238e+08,165702.0,17652.0,tonight woman audience trump town hall began q...,active,0.512500,0.483333,positive,76872.0,5,5
19865,3.836269e+07,74528.0,20615.0,america assembled joebiden kamalaharris barack...,less active,0.000000,0.000000,neutral,42180.2,4,4
94,1.398238e+08,59403.0,7489.0,god bette scared god fascist whitehouse racist...,active,0.450000,0.080000,positive,28254.6,4,4
90,1.398238e+08,59345.0,2838.0,wonderful speech tonight president vice presid...,active,0.680000,0.522500,positive,25440.8,4,4
92,1.398238e+08,45540.0,6133.0,god bette scared fascist whitehouse racist cri...,active,0.562500,0.100000,positive,21895.8,4,4
...,...,...,...,...,...,...,...,...,...,...,...
756,2.858792e+07,278.0,81.0,people think voting guy vote decency election ...,active,0.000000,0.000000,neutral,159.8,1,1
2640,4.439796e+07,278.0,64.0,joebiden celebrated abc right donaldtrump inte...,active,0.642857,0.317857,positive,149.6,1,1
12291,2.095488e+07,278.0,54.0,breaking joebiden take lead georgia vote trump...,active,0.650000,0.400000,positive,143.6,1,1
676,1.899340e+07,278.0,35.0,breaking abc news characterize joe biden appar...,active,0.412500,0.412500,positive,132.2,1,1


In [155]:
breaks = jenkspy.jenks_breaks(sorted_biden['retweet_count'], n_classes= num_bins)
print(breaks)
sorted_biden['retweet_count_category'] = pd.cut(sorted_biden['retweet_count'], bins=breaks, labels=range(1, num_bins+1), include_lowest=True)
sorted_biden.head(1000)

[0.0, 209.0, 1072.0, 2999.0, 7489.0, 20615.0]


,user_id,likes,retweet_count,cleantext,user_activity,subjectivity,polarity,analysis,interest,interest_category,likes_category,retweet_count_category
102,1.398238e+08,165702.0,17652.0,tonight woman audience trump town hall began q...,active,0.512500,0.483333,positive,76872.0,5,5,5
19865,3.836269e+07,74528.0,20615.0,america assembled joebiden kamalaharris barack...,less active,0.000000,0.000000,neutral,42180.2,4,4,5
94,1.398238e+08,59403.0,7489.0,god bette scared god fascist whitehouse racist...,active,0.450000,0.080000,positive,28254.6,4,4,4
90,1.398238e+08,59345.0,2838.0,wonderful speech tonight president vice presid...,active,0.680000,0.522500,positive,25440.8,4,4,3
92,1.398238e+08,45540.0,6133.0,god bette scared fascist whitehouse racist cri...,active,0.562500,0.100000,positive,21895.8,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...
756,2.858792e+07,278.0,81.0,people think voting guy vote decency election ...,active,0.000000,0.000000,neutral,159.8,1,1,1
2640,4.439796e+07,278.0,64.0,joebiden celebrated abc right donaldtrump inte...,active,0.642857,0.317857,positive,149.6,1,1,1
12291,2.095488e+07,278.0,54.0,breaking joebiden take lead georgia vote trump...,active,0.650000,0.400000,positive,143.6,1,1,1
676,1.899340e+07,278.0,35.0,breaking abc news characterize joe biden appar...,active,0.412500,0.412500,positive,132.2,1,1,1


In [156]:
sorted_biden.to_csv('sorted_biden_categorical.csv', index=False)

In [158]:
sorted_biden.groupby('likes_category')['polarity'].mean()

/var/folders/x5/k2wknhr57b1bk7btjrfpvfbm0000gn/T/ipykernel_97908/1352699997.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sorted_biden.groupby('likes_category')['polarity'].mean()


likes_category
1    0.080171
2    0.137957
3    0.187300
4    0.160500
5    0.483333
Name: polarity, dtype: float64

In [160]:
sorted_biden.groupby('retweet_count_category')['polarity'].mean()

/var/folders/x5/k2wknhr57b1bk7btjrfpvfbm0000gn/T/ipykernel_97908/3844585572.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sorted_biden.groupby('retweet_count_category')['polarity'].mean()


retweet_count_category
1    0.080197
2    0.088360
3    0.172905
4    0.131111
5    0.241667
Name: polarity, dtype: float64

Check how many tweets are in there for each group
Count negative and positive based on groups

In [161]:
sorted_biden.retweet_count_category.value_counts()

retweet_count_category
1    179183
2       349
3        32
4         9
5         2
Name: count, dtype: int64

In [163]:
sorted_biden.likes_category.value_counts()

likes_category
1    179401
2       152
3        16
4         5
5         1
Name: count, dtype: int64

In [164]:
sorted_biden.user_id.value_counts()

user_id
1.244982e+18    1259
8.742585e+08    1059
2.395433e+07     922
1.228206e+09     416
1.828681e+07     370
                ... 
2.894671e+09       1
9.531434e+07       1
1.238456e+18       1
1.234192e+08       1
1.199519e+18       1
Name: count, Length: 61201, dtype: int64

In [172]:
grouped_biden = sorted_biden.groupby(['user_id', 'analysis']).size().unstack(fill_value=0)
grouped_biden

analysis,negative,neutral,positive
user_id,,,
1.081000e+03,0,1,0
2.654000e+03,0,1,4
3.652000e+03,0,1,0
4.358000e+03,0,1,0
5.608000e+03,0,1,0
...,...,...,...
1.325492e+18,0,1,0
1.325502e+18,0,0,1
1.325533e+18,0,0,1


In [177]:
sorted_biden.groupby(['user_id']).size().sort_values(ascending=False)

user_id
1.244982e+18    1259
8.742585e+08    1059
2.395433e+07     922
1.228206e+09     416
1.828681e+07     370
                ... 
4.231909e+08       1
4.234418e+08       1
4.235975e+08       1
4.236203e+08       1
1.325550e+18       1
Length: 61201, dtype: int64